# Data Cleaning Notebook

This notebook documents the process of cleaning the raw tennis racquet listing data. Below is an overview of the steps I took (copied from `01_load_data.ipynb` takeaways section):


1. Get rid of all junior racquets from df

2. Add a racquet_brand column by extracting the brand name from racquet_name

3. Convert the following columns from object to float or int using regex or str logic:
    - Head Size
    - Length
    - Strung Weight
    - Balance
        - Create two columns: racquet_balance_in and racquet_balance_HH_HL
    - Stiffness
    - Beam width
    - String Pattern
        - Create two columns: racquet_mains and racquet_crosses
    - String Tension
        - Create two columns: racquet_tension_lower and racquet_tension_upper

## Table of Contents

1. [Imports and data loading](#imports-and-data-loading)
2. [Remove all junior racquets](#remove-all-junior-racquets)
3. [Drop extraneous columns](#drop-extraneous-columns)
4. [Convert spec columns](#convert-spec-columns)
5. [Final touchups](#final-touchups)
6. [Compare notebook and script outputs](#compare-notebook-and-script-outputs)

## Imports and data loading

In [1]:
# Import all libraries here
from pathlib import Path
import pandas as pd
import numpy as np
import re

In [2]:
# Setup data paths
DATA_DIR = Path().cwd().parent / "data"

raw_df = pd.read_csv(DATA_DIR / "raw" / "racquets_raw_2026_03_27.csv")

### Removing NA names and adding racquet brand

This is included as a subsection of the data loading section because it is low-effort and easy to implement. I also decided to extract the racquet SKU as a unique ID to allow for indexing and easy referencing between the embedded data and the metadata (this df).

In [3]:
# Create copy and drop NA names
df = raw_df.copy()
df = df.dropna(subset = ["racquet_name", "racquet_url"])

print(f"Removed {raw_df.shape[0] - df.shape[0]} rows.")
df.head()


Removed 20 rows.


,racquet_url,racquet_img,racquet_name,racquet_rating,racquet_rating_count,racquet_price,racquet_description,head_size,length,strung_weight,...,string_pattern,string_tension,racquet_colors_,racquet_colors_grey/,age,height,weight,other,age_9-10,strung _weight
0,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero 2026,3.8,6.0,299.0,Engineered for speed. With its fabled combinat...,100 in² / 645.16 cm²,27in / 68.58cm,11.2oz / 318g,...,"16 Mains / 19 Crosses Mains skip: 8T,8H Two Pi...",46-55 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero 98 2026,4.4,14.0,309.0,Speed. Spin. Accuracy. Featuring the smallest ...,98 in² / 632.26 cm²,27in / 68.58cm,11.4oz / 323g,...,"16 Mains / 20 Crosses Mains skip: 7T,9T,7H,9H ...",46-55 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero 98 2-Pack 2026,4.4,14.0,619.0,The most surgical member of the Pure Aero fami...,98 in² / 632.26 cm²,27in / 68.58cm,11.4oz / 323g,...,"16 Mains / 20 Crosses Mains skip: 7T,9T,7H,9H ...",46-55 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero Plus 2026,2.0,2.0,299.0,One of the games most powerful and spin-friend...,100 in² / 645.16 cm²,27.5in / 69.85cm,11.3oz / 320g,...,"16 Mains / 19 Crosses Mains skip: 8T,8H Two Pi...",46-55 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero Team 2026,4.0,1.0,279.0,Updated with improved aerodynamics and better ...,100 in² / 645.16 cm²,27in / 68.58cm,10.6oz / 301g,...,"16 Mains / 19 Crosses Mains skip: 7T,9T,7H,9H ...",44-53 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Extract Brand into its own column
df["racquet_brand"] = df["racquet_name"].apply(lambda x: x.split(" ")[0])

# Use SKU from URL to create an id column for indexing
df["racquet_id"] = df["racquet_url"].apply(lambda x: x.split("/")[::-1][0].split(".")[0].replace("descpage", ""))

# Check that each racquet has a UNIQUE id
assert df["racquet_id"].nunique() == df.shape[0]

In [5]:
# Reorder columns
first_cols = ["racquet_id", "racquet_url", "racquet_name", "racquet_brand"]
new_col_order = first_cols + [col for col in df.columns if col not in first_cols]
df = df[new_col_order]

## Remove all junior racquets

In this section, I will filter out all racquets that contain the word "Junior" in their name. These racquets seem to have NA values for most of their specs and are not relevent to my main goal of creating a semantic search product for avid tennis players (see `01_load_data.ipynb` for more of an explanation).

In [6]:
df_no_jr = df[~df["racquet_name"].str.lower().str.strip().str.contains("junior")]
print(f"Removed {df.shape[0] - df_no_jr.shape[0]} junior racquets.")

df.shape, df_no_jr.shape

Removed 83 junior racquets.


((418, 32), (335, 32))

In [7]:
df_no_jr.head()

,racquet_id,racquet_url,racquet_name,racquet_brand,racquet_img,racquet_rating,racquet_rating_count,racquet_price,racquet_description,head_size,...,string_pattern,string_tension,racquet_colors_,racquet_colors_grey/,age,height,weight,other,age_9-10,strung _weight
0,RCBAB-BPAR26,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,3.8,6.0,299.0,Engineered for speed. With its fabled combinat...,100 in² / 645.16 cm²,...,"16 Mains / 19 Crosses Mains skip: 8T,8H Two Pi...",46-55 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RCBAB-BPA98R,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero 98 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,4.4,14.0,309.0,Speed. Spin. Accuracy. Featuring the smallest ...,98 in² / 632.26 cm²,...,"16 Mains / 20 Crosses Mains skip: 7T,9T,7H,9H ...",46-55 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RCBAB-BPA982,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero 98 2-Pack 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,4.4,14.0,619.0,The most surgical member of the Pure Aero fami...,98 in² / 632.26 cm²,...,"16 Mains / 20 Crosses Mains skip: 7T,9T,7H,9H ...",46-55 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RCBAB-BPAPLR,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero Plus 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,2.0,2.0,299.0,One of the games most powerful and spin-friend...,100 in² / 645.16 cm²,...,"16 Mains / 19 Crosses Mains skip: 8T,8H Two Pi...",46-55 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,RCBAB-BPAT26,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero Team 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,4.0,1.0,279.0,Updated with improved aerodynamics and better ...,100 in² / 645.16 cm²,...,"16 Mains / 19 Crosses Mains skip: 7T,9T,7H,9H ...",44-53 pounds,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Drop extraneous columns

In this section, I drop the "extraneous" columns that were added to the dataframe when the scraper visited pages for special/non-standard or junior racquets (which will not be used in this search workflow). To remove these columns, I will filter out all columns with >95% NA values.

In [8]:
# List of cols to drop
cols_to_drop = []

for col in df_no_jr.columns:
    if df_no_jr[col].isna().sum() / df_no_jr.shape[0] > 0.95:
        cols_to_drop.append(col)
        
print("Drop the following columns:")
print("\n".join([f"- {col}" for col in cols_to_drop]))

Drop the following columns:
- racquet_colors_
- racquet_colors_grey/
- age
- height
- weight
- other
- age_9-10
- strung _weight


In [9]:
# Drop cols
df_dropped = df_no_jr.drop(columns = cols_to_drop)
df_dropped.head()

,racquet_id,racquet_url,racquet_name,racquet_brand,racquet_img,racquet_rating,racquet_rating_count,racquet_price,racquet_description,head_size,...,stiffness,beam_width,composition,power_level,stroke_style,swing_speed,racquet_colors,grip_type,string_pattern,string_tension
0,RCBAB-BPAR26,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,3.8,6.0,299.0,Engineered for speed. With its fabled combinat...,100 in² / 645.16 cm²,...,66,23mm / 26mm / 23mm,Graphite,Low-Medium,Medium-Full,Medium-Fast,Yellow/Black,Babolat Syntec Pro,"16 Mains / 19 Crosses Mains skip: 8T,8H Two Pi...",46-55 pounds
1,RCBAB-BPA98R,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero 98 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,4.4,14.0,309.0,Speed. Spin. Accuracy. Featuring the smallest ...,98 in² / 632.26 cm²,...,66,21mm / 23mm / 22mm,Graphite,Low-Medium,Medium-Full,Medium-Fast,Grey/Yellow,Syntec Pro,"16 Mains / 20 Crosses Mains skip: 7T,9T,7H,9H ...",46-55 pounds
3,RCBAB-BPA982,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero 98 2-Pack 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,4.4,14.0,619.0,The most surgical member of the Pure Aero fami...,98 in² / 632.26 cm²,...,66,21mm / 23mm / 22mm,Graphite,Low-Medium,Medium-Full,Medium-Fast,Black/Yellow,Syntec Pro,"16 Mains / 20 Crosses Mains skip: 7T,9T,7H,9H ...",46-55 pounds
4,RCBAB-BPAPLR,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero Plus 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,2.0,2.0,299.0,One of the games most powerful and spin-friend...,100 in² / 645.16 cm²,...,65,23mm / 26mm / 23mm,Graphite,Low-Medium,Medium-Full,Medium-Fast,Grey/Yellow,Babolat Syntec Pro,"16 Mains / 19 Crosses Mains skip: 8T,8H Two Pi...",46-55 pounds
5,RCBAB-BPAT26,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero Team 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,4.0,1.0,279.0,Updated with improved aerodynamics and better ...,100 in² / 645.16 cm²,...,66,23mm / 26mm / 23mm,Graphite,Low-Medium,Medium-Full,Medium-Fast,Grey/Yellow,Babolat Syntec Pro,"16 Mains / 19 Crosses Mains skip: 7T,9T,7H,9H ...",44-53 pounds


In [12]:
# Assert that 6 cols were dropped
assert df_no_jr.shape[1] == df_dropped.shape[1] + len(cols_to_drop)

# Assert that only age, weight, height, other, age_9-10, and strung_weight were dropped
assert set(df_no_jr.columns) - set(df_dropped.columns) == set(cols_to_drop)

I will save each step of the data cleaning process to the "interim" folder to document the cleaning process (and maybe for future reference).

In [13]:
# Save to data/interim
df_dropped.to_csv(DATA_DIR / "interim" / "racquets_staging_2026_03_27.csv")

## Convert spec columns

In this section I convert the specified columns from `str` to `float`. 

Columns to convert:

- head_size
- length
- strung_weight
- balance
    - Create two columns: racquet_balance_in and racquet_balance_HH_HL
- stiffness
- beam_width
- string_pattern
    - Create two columns: racquet_mains and racquet_crosses
- string_tension
    - Create two columns: racquet_tension_lower and racquet_tension_upper

In [14]:
# Create a copy of df for regex ops
df_regex = df_dropped.copy()

# Get cols to convert
str_cols = []
for col in df_regex.columns:
    if df_regex[col].dtype == "object" and "racquet_" not in col:
        str_cols.append(col)
    
    else:
        pass
    
print(str_cols)

['head_size', 'length', 'strung_weight', 'balance', 'stiffness', 'beam_width', 'composition', 'power_level', 'stroke_style', 'swing_speed', 'grip_type', 'string_pattern', 'string_tension']


### Extract head size

In [15]:
# Extract head size using units
df_regex["racquet_head_size_sq_in"] = (
    df_regex["head_size"]\
        .str.extract(r"(\d+\.?\d*)\s*(?:in²|in|sq\s*in)")\
            .astype(float)
)

df_regex[["head_size", "racquet_head_size_sq_in"]]

,head_size,racquet_head_size_sq_in
0,100 in² / 645.16 cm²,100.0
1,98 in² / 632.26 cm²,98.0
3,98 in² / 632.26 cm²,98.0
4,100 in² / 645.16 cm²,100.0
5,100 in² / 645.16 cm²,100.0
...,...,...
433,98 in² / 632.26 cm²,98.0
434,98 in² / 632.26 cm²,98.0
435,98 in² / 632.26 cm²,98.0
436,100 in² / 645.16 cm²,100.0


### Extract racquet length

In [16]:
# Use suffix to get length
df_regex["racquet_length_in"] = (
    df_regex["length"]
    .str.extract(r"(\d+\.?\d*)\s*(?:in²|in|sq\s*in)")
    .astype(float)
)

df_regex[["length", "racquet_length_in"]]

,length,racquet_length_in
0,27in / 68.58cm,27.0
1,27in / 68.58cm,27.0
3,27in / 68.58cm,27.0
4,27.5in / 69.85cm,27.5
5,27in / 68.58cm,27.0
...,...,...
433,27.5in / 69.85cm,27.5
434,28in / 71.12cm,28.0
435,27.5in / 69.85cm,27.5
436,27in / 68.58cm,27.0


### Extract racquet strung weight in ounces

In [17]:
# Extract strung weight with suffix
df_regex["racquet_strung_weight_oz"] = (
    df_regex["strung_weight"]
    .str.extract(r"(\d+\.?d*)\s*")
    .astype(float)
)

df_regex[["strung_weight", "racquet_strung_weight_oz"]]

,strung_weight,racquet_strung_weight_oz
0,11.2oz / 318g,11.0
1,11.4oz / 323g,11.0
3,11.4oz / 323g,11.0
4,11.3oz / 320g,11.0
5,10.6oz / 301g,10.0
...,...,...
433,11.4oz / 323g,11.0
434,11.4oz / 323g,11.0
435,11.4oz / 323g,11.0
436,11.1oz / 315g,11.0


### Extract racquet balance (in)

In [18]:
# Extract racquet balance in inches by matching digits to in suffix
df_regex["racquet_balance_in"] = (
    df_regex["balance"]\
    .str.extract(r"(\d+(?:\.\d+)?)\s*in\b").
    astype(float)
)

# Extract Balance number and label separately
extracted = df_regex["balance"].str.extract(
    r"(\d+(?:\.\d+)?)\s*(?:pts\s*)?(HL|HH|EB)\b"
)

extracted.columns = ["value", "label"]
extracted["value"] = extracted["value"].astype(float)

# If HL, make pos; If HH, make neg
def apply_balance_sign(row):
    if row["label"] == "HL":
        return row['value']
    
    elif row["label"] == "HH":
        return -row["value"]
    
    elif row["label"] == "EB":
        return 0.0
    
    return None

# Apply function to extracted df and save into new col in df_regex
df_regex["racquet_balance_HH_HL"] = extracted.apply(apply_balance_sign, axis = 1)

df_regex[["balance", "racquet_balance_in", "racquet_balance_HH_HL"]]

,balance,racquet_balance_in,racquet_balance_HH_HL
0,12.99in / 32.99cm / 4 pts HL,12.99,4.0
1,12.79in / 32.49cm / 6 pts HL,12.79,6.0
3,12.79in / 32.49cm / 6 pts HL,12.79,6.0
4,12.99in / 32.99cm / 6 pts HL,12.99,6.0
5,13in / 33.02cm / 4 pts HL,13.00,4.0
...,...,...,...
433,13in / 33.02cm / 6 pts HL,13.00,6.0
434,13in / 33.02cm / 8 pts HL,13.00,8.0
435,13in / 33.02cm / 6 pts HL,13.00,6.0
436,12.9in / 32.77cm / 5 pts HL,12.90,5.0


### Extract racquet stiffness

In [19]:
# Extract stiffness by replacing str NAs and casting to float
df_regex["racquet_stiffness"] = df_regex["stiffness"]
df_regex['racquet_stiffness'] = df_regex['racquet_stiffness'].replace('N/A (very low)', np.nan)

df_regex["racquet_stiffness"] = df_regex["racquet_stiffness"].astype(float)

df_regex[["stiffness", "racquet_stiffness"]] # Display

,stiffness,racquet_stiffness
0,66,66.0
1,66,66.0
3,66,66.0
4,65,65.0
5,66,66.0
...,...,...
433,65,65.0
434,64,64.0
435,67,67.0
436,69,69.0


### Extract average racquet beam width

Because beam width is measured in three areas, I decided to create a new column to store the average beam width of each racquet.

In [20]:
# Get average beam width as proxy for 3-value beam width field
def average_beam_width(value):
    if isinstance(value, str):
        parts = value.split("/") # List with 3 strs
        numbers = [] # Collect three width values
        
        # For each item in parts, strip/replace and then append #s
        for part in parts:
            cleaned = part.strip().replace("mm", "") # Remove suffix
            
            if cleaned:
                try:
                    numbers.append(float(cleaned))
                    
                except ValueError:
                    pass
        
        # If list is not empty, calc avg -> Else return nan      
        if numbers:
            return sum(numbers) / len(numbers)
        
        else:
            return float("nan")
        
    else:
        return float("nan") # Return nan if beam width isn't a str
    
df_regex["racquet_avg_beam_width"] = df_regex["beam_width"].apply(average_beam_width)

df_regex[["beam_width", "racquet_avg_beam_width"]]

,beam_width,racquet_avg_beam_width
0,23mm / 26mm / 23mm,24.000000
1,21mm / 23mm / 22mm,22.000000
3,21mm / 23mm / 22mm,22.000000
4,23mm / 26mm / 23mm,24.000000
5,23mm / 26mm / 23mm,24.000000
...,...,...
433,21.7mm / 21.7mm / 21.7mm,21.700000
434,21.7mm / 21.7mm / 21.7mm,21.700000
435,21.7mm / 21.7mm / 21.7mm,21.700000
436,23mm / 25mm / 23mm,23.666667


### Extract racquet mains and crosses

In [21]:
# Extract main and cross values, assign to relevant column in series, and pass series to df
def extract_mains_crosses(value):
    mains = np.nan
    crosses = np.nan
    
    if isinstance(value, str) and value.strip():
        mains_regex = re.search(r'(\d+)\s*Mains', value, re.IGNORECASE)
        crosses_regex = re.search(r'(\d+)\s*Crosses', value, re.IGNORECASE)
        
        if mains_regex:
            mains = float(mains_regex.group(1))
            
        if crosses_regex:
            crosses = float(crosses_regex.group(1))
            
    return pd.Series([mains, crosses])


df_regex[["racquet_mains", "racquet_crosses"]] = df_regex["string_pattern"].apply(extract_mains_crosses)

df_regex[["string_pattern", "racquet_mains", "racquet_crosses"]]

,string_pattern,racquet_mains,racquet_crosses
0,"16 Mains / 19 Crosses Mains skip: 8T,8H Two Pi...",16.0,19.0
1,"16 Mains / 20 Crosses Mains skip: 7T,9T,7H,9H ...",16.0,20.0
3,"16 Mains / 20 Crosses Mains skip: 7T,9T,7H,9H ...",16.0,20.0
4,"16 Mains / 19 Crosses Mains skip: 8T,8H Two Pi...",16.0,19.0
5,"16 Mains / 19 Crosses Mains skip: 7T,9T,7H,9H ...",16.0,19.0
...,...,...,...
433,"18 Mains / 20 Crosses Mains skip: 8T,10T,8H,10...",18.0,20.0
434,"16 Mains / 19 Crosses Mains skip: 7T,9T,7H,9H ...",16.0,19.0
435,"16 Mains / 19 Crosses Mains skip: 7T,9T,7H,9H ...",16.0,19.0
436,"16 Mains / 19 Crosses Mains skip: 7T,9T,7H,9H ...",16.0,19.0


### Extract racquet tension

In [22]:
# Extract tension bounds by splitting with regex and casting to float
def extract_tension_bounds(value):
    # Init as nan -> if can't extract returns nan
    lower = np.nan
    upper = np.nan
    
    if isinstance(value, str) and value.strip():
        tension_regex = re.search(r'(\d+)\s*-\s*(\d+)', value)
        
        if tension_regex:
            lower = float(tension_regex.group(1))
            upper = float(tension_regex.group(2))
        
    return pd.Series([lower, upper])

df_regex[["racquet_tension_lower", "racquet_tension_upper"]] = df_regex["string_tension"].apply(extract_tension_bounds)

df_regex[["string_tension", "racquet_tension_lower", "racquet_tension_upper"]]

,string_tension,racquet_tension_lower,racquet_tension_upper
0,46-55 pounds,46.0,55.0
1,46-55 pounds,46.0,55.0
3,46-55 pounds,46.0,55.0
4,46-55 pounds,46.0,55.0
5,44-53 pounds,44.0,53.0
...,...,...,...
433,45-55 pounds,45.0,55.0
434,45-55 pounds,45.0,55.0
435,45-55 pounds,45.0,55.0
436,51-55 pounds,51.0,55.0


## Final touchups

After casting the above columns, I can now drop the old columns and rename the unchanged columns to match my naming scheme (racquet_<feature>).

In [23]:
df_regex.columns

Index(['racquet_id', 'racquet_url', 'racquet_name', 'racquet_brand',
       'racquet_img', 'racquet_rating', 'racquet_rating_count',
       'racquet_price', 'racquet_description', 'head_size', 'length',
       'strung_weight', 'balance', 'swingweight', 'stiffness', 'beam_width',
       'composition', 'power_level', 'stroke_style', 'swing_speed',
       'racquet_colors', 'grip_type', 'string_pattern', 'string_tension',
       'racquet_head_size_sq_in', 'racquet_length_in',
       'racquet_strung_weight_oz', 'racquet_balance_in',
       'racquet_balance_HH_HL', 'racquet_stiffness', 'racquet_avg_beam_width',
       'racquet_mains', 'racquet_crosses', 'racquet_tension_lower',
       'racquet_tension_upper'],
      dtype='object')

In [24]:
# Copy before dropping old cols and renaming
df_cleaned = df_regex.copy()

# Drop old cols
cols_to_drop = ["head_size", "length", "strung_weight", "balance", 
                "beam_width", "string_pattern", "string_tension", "stiffness"]
df_cleaned = df_cleaned.drop(columns = cols_to_drop)

# Rename remaining cols
df_cleaned = df_cleaned.rename(columns = {"swingweight" : "racquet_swingweight",
                                  "composition" : "racquet_composition",
                                  "power_level" : "racquet_power_level",
                                  "stroke_style" : "racquet_stroke_style",
                                  "swing_speed" : "racquet_swing_speed",
                                  "racquet_colors" : "racquet_colors",
                                  "grip_type" : "racquet_grip_type"
                                  })

In [25]:
df_cleaned.columns

Index(['racquet_id', 'racquet_url', 'racquet_name', 'racquet_brand',
       'racquet_img', 'racquet_rating', 'racquet_rating_count',
       'racquet_price', 'racquet_description', 'racquet_swingweight',
       'racquet_composition', 'racquet_power_level', 'racquet_stroke_style',
       'racquet_swing_speed', 'racquet_colors', 'racquet_grip_type',
       'racquet_head_size_sq_in', 'racquet_length_in',
       'racquet_strung_weight_oz', 'racquet_balance_in',
       'racquet_balance_HH_HL', 'racquet_stiffness', 'racquet_avg_beam_width',
       'racquet_mains', 'racquet_crosses', 'racquet_tension_lower',
       'racquet_tension_upper'],
      dtype='object')

In [26]:
# Save cleaned df to data/interim
df_cleaned.to_csv(DATA_DIR / "interim" / "racquets_cleaned_2026_03_27.csv")

## Compare notebook and script outputs

This section is just a quick check to make sure that the process followed through this notebook and the output of `scripts/clean_raw_data.py` are the same.

**After checking**: .equals() returns false because their columns are in different orders. I don't reorder the columns in the cleaning script (only did that here for readability).

In [ ]:
# Just for experiment I renamed NB output with a _nb at the end 
# and then ran the script to get the script-cleaned data
df_cleaned_script = pd.read_csv(DATA_DIR / "interim" / "racquets_cleaned_2026_03_27.csv", index_col = 0)

In [50]:
df_cleaned.equals(df_cleaned_script)

False

In [53]:
df_cleaned.index, df_cleaned_script.index

(Index([  0,   1,   3,   4,   5,   6,   7,   8,   9,  10,
        ...
        428, 429, 430, 431, 432, 433, 434, 435, 436, 437],
       dtype='int64', length=335),
 Index([  0,   1,   3,   4,   5,   6,   7,   8,   9,  10,
        ...
        428, 429, 430, 431, 432, 433, 434, 435, 436, 437],
       dtype='int64', length=335))

In [41]:
df_cleaned.columns, len(df_cleaned.columns)

(Index(['racquet_id', 'racquet_url', 'racquet_name', 'racquet_brand',
        'racquet_img', 'racquet_rating', 'racquet_rating_count',
        'racquet_price', 'racquet_description', 'racquet_swingweight',
        'racquet_composition', 'racquet_power_level', 'racquet_stroke_style',
        'racquet_swing_speed', 'racquet_colors', 'racquet_grip_type',
        'racquet_head_size_sq_in', 'racquet_length_in',
        'racquet_strung_weight_oz', 'racquet_balance_in',
        'racquet_balance_HH_HL', 'racquet_stiffness', 'racquet_avg_beam_width',
        'racquet_mains', 'racquet_crosses', 'racquet_tension_lower',
        'racquet_tension_upper'],
       dtype='object'),
 27)

In [42]:
df_cleaned_script.columns, len(df_cleaned_script.columns)

(Index(['racquet_url', 'racquet_img', 'racquet_name', 'racquet_rating',
        'racquet_rating_count', 'racquet_price', 'racquet_description',
        'racquet_swingweight', 'racquet_composition', 'racquet_power_level',
        'racquet_stroke_style', 'racquet_swing_speed', 'racquet_colors',
        'racquet_grip_type', 'racquet_brand', 'racquet_id',
        'racquet_head_size_sq_in', 'racquet_length_in',
        'racquet_strung_weight_oz', 'racquet_balance_in',
        'racquet_balance_HH_HL', 'racquet_stiffness', 'racquet_avg_beam_width',
        'racquet_mains', 'racquet_crosses', 'racquet_tension_lower',
        'racquet_tension_upper'],
       dtype='object'),
 27)

In [43]:
df_cleaned.shape, df_cleaned_script.shape

((335, 27), (335, 27))

In [49]:
for col in list(df_cleaned.columns):
    if df_cleaned[col].dtype == df_cleaned_script[col].dtype:
       print(f"Dtypes of column {col} match")
    else:
        print(f"Dtypes of column {col} don't match.")
        break 


Dtypes of column racquet_id match
Dtypes of column racquet_url match
Dtypes of column racquet_name match
Dtypes of column racquet_brand match
Dtypes of column racquet_img match
Dtypes of column racquet_rating match
Dtypes of column racquet_rating_count match
Dtypes of column racquet_price match
Dtypes of column racquet_description match
Dtypes of column racquet_swingweight match
Dtypes of column racquet_composition match
Dtypes of column racquet_power_level match
Dtypes of column racquet_stroke_style match
Dtypes of column racquet_swing_speed match
Dtypes of column racquet_colors match
Dtypes of column racquet_grip_type match
Dtypes of column racquet_head_size_sq_in match
Dtypes of column racquet_length_in match
Dtypes of column racquet_strung_weight_oz match
Dtypes of column racquet_balance_in match
Dtypes of column racquet_balance_HH_HL match
Dtypes of column racquet_stiffness match
Dtypes of column racquet_avg_beam_width match
Dtypes of column racquet_mains match
Dtypes of column ra

In [54]:
df_cleaned.head()

,racquet_id,racquet_url,racquet_name,racquet_brand,racquet_img,racquet_rating,racquet_rating_count,racquet_price,racquet_description,racquet_swingweight,...,racquet_length_in,racquet_strung_weight_oz,racquet_balance_in,racquet_balance_HH_HL,racquet_stiffness,racquet_avg_beam_width,racquet_mains,racquet_crosses,racquet_tension_lower,racquet_tension_upper
0,RCBAB-BPAR26,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,3.8,6.0,299.0,Engineered for speed. With its fabled combinat...,320.0,...,27.0,11.0,12.99,4.0,66.0,24.0,16.0,19.0,46.0,55.0
1,RCBAB-BPA98R,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero 98 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,4.4,14.0,309.0,Speed. Spin. Accuracy. Featuring the smallest ...,322.0,...,27.0,11.0,12.79,6.0,66.0,22.0,16.0,20.0,46.0,55.0
3,RCBAB-BPA982,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero 98 2-Pack 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,4.4,14.0,619.0,The most surgical member of the Pure Aero fami...,322.0,...,27.0,11.0,12.79,6.0,66.0,22.0,16.0,20.0,46.0,55.0
4,RCBAB-BPAPLR,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero Plus 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,2.0,2.0,299.0,One of the games most powerful and spin-friend...,330.0,...,27.5,11.0,12.99,6.0,65.0,24.0,16.0,19.0,46.0,55.0
5,RCBAB-BPAT26,https://www.tennis-warehouse.com/Babolat_Pure_...,Babolat Pure Aero Team 2026,Babolat,https://img.tennis-warehouse.com/watermark/rs....,4.0,1.0,279.0,Updated with improved aerodynamics and better ...,306.0,...,27.0,10.0,13.00,4.0,66.0,24.0,16.0,19.0,44.0,53.0


In [ ]:
df_cleaned_script.head() # Reason why they don't match is just column order -> that's fine

,racquet_url,racquet_img,racquet_name,racquet_rating,racquet_rating_count,racquet_price,racquet_description,racquet_swingweight,racquet_composition,racquet_power_level,...,racquet_length_in,racquet_strung_weight_oz,racquet_balance_in,racquet_balance_HH_HL,racquet_stiffness,racquet_avg_beam_width,racquet_mains,racquet_crosses,racquet_tension_lower,racquet_tension_upper
0,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero 2026,3.8,6.0,299.0,Engineered for speed. With its fabled combinat...,320.0,Graphite,Low-Medium,...,27.0,11.2,12.99,4.0,66.0,24.0,16.0,19.0,46.0,55.0
1,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero 98 2026,4.4,14.0,309.0,Speed. Spin. Accuracy. Featuring the smallest ...,322.0,Graphite,Low-Medium,...,27.0,11.4,12.79,6.0,66.0,22.0,16.0,20.0,46.0,55.0
3,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero 98 2-Pack 2026,4.4,14.0,619.0,The most surgical member of the Pure Aero fami...,322.0,Graphite,Low-Medium,...,27.0,11.4,12.79,6.0,66.0,22.0,16.0,20.0,46.0,55.0
4,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero Plus 2026,2.0,2.0,299.0,One of the games most powerful and spin-friend...,330.0,Graphite,Low-Medium,...,27.5,11.3,12.99,6.0,65.0,24.0,16.0,19.0,46.0,55.0
5,https://www.tennis-warehouse.com/Babolat_Pure_...,https://img.tennis-warehouse.com/watermark/rs....,Babolat Pure Aero Team 2026,4.0,1.0,279.0,Updated with improved aerodynamics and better ...,306.0,Graphite,Low-Medium,...,27.0,10.6,13.00,4.0,66.0,24.0,16.0,19.0,44.0,53.0
